In [8]:
!pip install -q einops

import os
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
from torchvision import transforms
from einops import rearrange

torch.manual_seed(42)
np.random.seed(42)

print("Environment setup")
print("GPU:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

Environment setup
GPU: False
GPU count: 0


Patch Embedding

In [9]:
class PatchEmbedding(nn.Module):

    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)
        x = rearrange(x, "b c h w -> b (h w) c")
        return x

Transformer Block

In [10]:
class TransformerBlock(nn.Module):

    def __init__(self, dim, num_heads, mlp_ratio=4.0, dropout=0.0, attn_dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            dim,
            num_heads,
            dropout=attn_dropout,
            batch_first=True
        )
        self.norm2 = nn.LayerNorm(dim)
        hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x

Vision Transformer

In [11]:
class VisionTransformer(nn.Module):

    def __init__(self,
                 img_size=224,
                 patch_size=16,
                 in_channels=3,
                 embed_dim=768,
                 depth=12,
                 num_heads=12):

        super().__init__()
        self.embed_dim = embed_dim
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_embed = PatchEmbedding(
            img_size,
            patch_size,
            in_channels,
            embed_dim
        )

        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.num_patches, embed_dim)
        )

        self.blocks = nn.ModuleList(
            [TransformerBlock(embed_dim, num_heads) for _ in range(depth)]
        )

        self.norm = nn.LayerNorm(embed_dim)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x, mask_tokens_indices=None):
        if x.dim() == 4:
            x = self.patch_embed(x)
        if mask_tokens_indices is not None:
            if isinstance(mask_tokens_indices, np.ndarray):
                mask_tokens_indices = torch.from_numpy(mask_tokens_indices).long()
            elif not isinstance(mask_tokens_indices, torch.Tensor):
                mask_tokens_indices = torch.tensor(mask_tokens_indices).long()
            mask_tokens_indices = mask_tokens_indices.to(self.pos_embed.device)
            B = x.shape[0]
            pos_embed = self.pos_embed.expand(B, -1, -1)
            idx = mask_tokens_indices.unsqueeze(-1).expand(-1, -1, self.embed_dim)
            pos_embed = torch.gather(pos_embed, dim=1, index=idx)
        else:
            pos_embed = self.pos_embed

        x = x + pos_embed
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return x

Encoder

In [12]:
class MAEEncoder(nn.Module):

    def __init__(self,
                 img_size=224,
                 patch_size=16,
                 embed_dim=768,
                 depth=12,
                 num_heads=12):

        super().__init__()

        print("Encoder created")

        self.vit = VisionTransformer(
            img_size=img_size,
            patch_size=patch_size,
            embed_dim=embed_dim,
            depth=depth,
            num_heads=num_heads
        )

    def forward(self, x, mask_tokens_indices):
        return self.vit(x, mask_tokens_indices)

Decoder

In [13]:
class MAEDecoder(nn.Module):

    def __init__(self,
                 num_patches=196,
                 patch_size=16,
                 embed_dim=768,
                 depth=12,
                 num_heads=6,
                 decoder_embed_dim=384):

        super().__init__()

        print("Decoder created")

        self.num_patches = num_patches
        self.decoder_embed_dim = decoder_embed_dim
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim)
        self.mask_token = nn.Parameter(
            torch.zeros(1, 1, decoder_embed_dim)
        )

        self.decoder_pos_embed = nn.Parameter(
            torch.zeros(1, num_patches, decoder_embed_dim)
        )

        self.decoder_blocks = nn.ModuleList(
            [TransformerBlock(decoder_embed_dim, num_heads) for _ in range(depth)]
        )

        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)

        self.pred_head = nn.Linear(
            decoder_embed_dim,
            patch_size * patch_size * 3
        )

    def forward(self, encoder_latent, visible_indices):
        B = encoder_latent.shape[0]
        x = self.decoder_embed(encoder_latent)
        if isinstance(visible_indices, np.ndarray):
            visible_indices = torch.from_numpy(visible_indices).long()
        elif not isinstance(visible_indices, torch.Tensor):
            visible_indices = torch.tensor(visible_indices).long()
        visible_indices = visible_indices.to(x.device)
        if visible_indices.dim() == 1:
            visible_indices = visible_indices.unsqueeze(0).expand(B, -1)
        mask_tokens = self.mask_token.expand(B, self.num_patches, -1).to(dtype=x.dtype)
        idx = visible_indices.unsqueeze(-1).expand(-1, -1, self.decoder_embed_dim)
        full_seq = mask_tokens.scatter(dim=1, index=idx, src=x)
        full_seq = full_seq + self.decoder_pos_embed
        for block in self.decoder_blocks:
            full_seq = block(full_seq)
        full_seq = self.decoder_norm(full_seq)
        return self.pred_head(full_seq)

MAE

In [14]:
class MaskedAutoencoder(nn.Module):

    def __init__(self,
                 img_size=224,
                 patch_size=16,
                 mask_ratio=0.75,
                 encoder_embed_dim=768,
                 encoder_depth=12,
                 encoder_num_heads=12,
                 decoder_embed_dim=384,
                 decoder_depth=12,
                 decoder_num_heads=6):

        super().__init__()

        self.img_size = img_size
        self.patch_size = patch_size
        self.mask_ratio = mask_ratio

        num_patches = (img_size // patch_size) ** 2

        self.num_patches = num_patches
        self.num_visible = int(num_patches * (1 - mask_ratio))
        self.num_masked = num_patches - self.num_visible
        self.patch_embed = PatchEmbedding(
            img_size,
            patch_size,
            3,
            encoder_embed_dim
        )

        self.encoder = MAEEncoder(
            img_size,
            patch_size,
            encoder_embed_dim,
            encoder_depth,
            encoder_num_heads
        )

        self.decoder = MAEDecoder(
            num_patches,
            patch_size,
            encoder_embed_dim,
            decoder_depth,
            decoder_num_heads,
            decoder_embed_dim
        )

Patch Functions

In [15]:
    def patchify(self, x):

        p = self.patch_size
        return rearrange(
            x,
            "b c (h p1) (w p2) -> b (h w) (p1 p2 c)",
            p1=p,
            p2=p
        )

    def unpatchify(self, x):
        p = self.patch_size
        h = w = int(self.num_patches ** 0.5)
        return rearrange(
            x,
            "b (h w) (p1 p2 c) -> b c (h p1) (w p2)",
            h=h,
            w=w,
            p1=p,
            p2=p,
            c=3
        )

Masking

In [16]:
    def random_masking(self, x):

        B = x.shape[0]
        noise = torch.rand(B, self.num_patches, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        mask = torch.ones(B, self.num_patches, device=x.device)
        mask[:, self.num_visible:] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)
        ids_keep = ids_shuffle[:, :self.num_visible]
        x_visible = torch.gather(
            x,
            dim=1,
            index=ids_keep.unsqueeze(-1).expand(-1, -1, x.shape[-1])
        )

        return x_visible, mask, ids_keep

Forward

In [17]:
    def forward(self, x):
        patches = self.patch_embed(x)
        x_visible, mask, visible_indices = self.random_masking(patches)
        encoder_latent = self.encoder(x_visible, visible_indices)
        pred = self.decoder(encoder_latent, visible_indices)
        return pred, mask

Loss

In [18]:
    def forward_loss(self, imgs, pred, mask):
        target = self.patchify(imgs)
        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)
        loss = (loss * (1 - mask)).sum() / (1 - mask).sum()
        return loss

Weight Initialization

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.trunc_normal_(m.weight, std=0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

    elif isinstance(m, nn.LayerNorm):
        nn.init.constant_(m.bias, 0)
        nn.init.constant_(m.weight, 1.0)

print("Model classes ready")

Dataset

In [19]:
class TinyImageNetDataset(Dataset):

    def __init__(self, root_dir, split="train", transform=None):
        self.root_dir = Path(root_dir)
        self.split = split
        self.transform = transform
        self.samples = []
        self.load_data()

    def load_data(self):
        data_dir = self.root_dir / ("train" if self.split == "train" else "val")
        if self.split == "train":
            for class_dir in sorted(data_dir.iterdir())[:100]:
                images_dir = class_dir / "images"
                for img_path in images_dir.glob("*.JPEG"):
                    self.samples.append(img_path)

        else:

            images_dir = data_dir / "images"
            for img_path in list(images_dir.glob("*.JPEG"))[:2000]:
                self.samples.append(img_path)

        print(self.split, "images:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        try:

            image = Image.open(self.samples[idx]).convert("RGB")
            if self.transform:
                image = self.transform(image)

            return image

        except:
            return torch.randn(3, 224, 224)

Data Transforms

In [20]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.4, 0.4, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

Load Dataset

In [ ]:
dataset_path = Path("/kaggle/input/datasets/akash2sharma/tiny-imagenet/tiny-imagenet-200")

train_dataset = TinyImageNetDataset(dataset_path,"train",train_transform)

val_dataset = TinyImageNetDataset(dataset_path,"val",val_transform)

train_loader = DataLoader(train_dataset,
                          batch_size=32,
                          shuffle=True,
                          num_workers=4,
                          pin_memory=True)

val_loader = DataLoader(val_dataset,
                        batch_size=32,
                        shuffle=False,
                        num_workers=4,
                        pin_memory=True)

print("Train batches:",len(train_loader))
print("Val batches:",len(val_loader))

Training Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MaskedAutoencoder()
model = model.to(device)
total_params = sum(p.numel() for p in model.parameters())
print("Total parameters:",total_params)

Optimizer

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=len(train_loader)*20,
    eta_min=1e-6
)

scaler = GradScaler()

Training Loop

In [ ]:
num_epochs = 20

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for images in tqdm(train_loader):
        images = images.to(device)
        optimizer.zero_grad()

        with autocast():
            pred,mask = model(images)
            loss = model.forward_loss(images,pred,mask)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    model.eval()

    val_loss = 0
    with torch.no_grad():
        for images in val_loader:

            images = images.to(device)
            pred,mask = model(images)
            loss = model.forward_loss(images,pred,mask)
            val_loss += loss.item()

    val_loss /= len(val_loader)
    val_losses.append(val_loss)

    print("Epoch",epoch+1,
          "Train:",train_loss,
          "Val:",val_loss)

Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label="Train Loss", linewidth=2.5, marker="o", markersize=4)
axes[0].plot(val_losses, label="Val Loss", linewidth=2.5, marker="s", markersize=4)
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Loss", fontsize=12)
axes[0].set_title("Training and Validation Loss", fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(train_losses, label="Train Loss", linewidth=2.5, marker="o", markersize=4)
axes[1].semilogy(val_losses, label="Val Loss", linewidth=2.5, marker="s", markersize=4)
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Loss (log scale)", fontsize=12)
axes[1].set_title("Training and Validation Loss (Log Scale)", fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, which="both")

plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print("Final training loss:", f"{train_losses[-1]:.6f}")
print("Final validation loss:", f"{val_losses[-1]:.6f}")

Evaluation Metrics

In [ ]:
def denormalize(x):

    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1).to(x.device)
    std = torch.tensor([0.229,0.224,0.225]).view(3,1,1).to(x.device)
    return torch.clamp(x*std+mean,0,1)

def compute_psnr(original,reconstructed):

    mse = torch.mean((original-reconstructed)**2)
    if mse < 1e-10:
        return 100

    return (20*torch.log10(1.0/torch.sqrt(mse))).item()

def compute_ssim(original,reconstructed):

    o = original.reshape(-1)
    r = reconstructed.reshape(-1)
    num = ((o-o.mean())*(r-r.mean())).mean()
    den = o.std()*r.std()

    if den < 1e-10:
        return 1

    return (num/den).item()

Reconstruction and Visualization

In [ ]:
test_images = next(iter(val_loader)).to(device)

model.eval()
with torch.no_grad():
    pred, mask = model(test_images)
    target = model.patchify(test_images)

    masked_patches = target.clone()
    masked_patches[mask == 0] = 0

    masked_images = model.unpatchify(masked_patches)
    reconstructed = model.unpatchify(pred)

    fig, axes = plt.subplots(5, 3, figsize=(15, 25))

psnr_scores = []
ssim_scores = []

for i in range(5):
    orig = denormalize(test_images[i]).cpu()
    mask_img = denormalize(masked_images[i]).cpu()
    recon = denormalize(reconstructed[i]).cpu()

    psnr = compute_psnr(orig, recon)
    ssim = compute_ssim(orig, recon)

    psnr_scores.append(psnr)
    ssim_scores.append(ssim)

    axes[i, 0].imshow(orig.permute(1, 2, 0).numpy())
    axes[i, 0].set_title("Original")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask_img.permute(1, 2, 0).numpy())
    axes[i, 1].set_title(f"Masked (75%)\nPSNR: {psnr:.2f} dB")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(recon.permute(1, 2, 0).numpy())
    axes[i, 2].set_title(f"Reconstructed\nSSIM: {ssim:.4f}")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/reconstruction_samples.png", dpi=150, bbox_inches="tight")
plt.show()

Summary

In [ ]:
print()
print("Assignment complete")
print()
print("Summary:")
print("Part 1: Patchification and Masking")
print("Part 2: Forward Pass")
print("Part 3: Training")
print("Part 4: Evaluation")
print()
print("Final Metrics:")
print("Train Loss :", f"{train_losses[-1]:.6f}")
print("Val Loss   :", f"{val_losses[-1]:.6f}")
print("Avg PSNR   :", f"{np.mean(psnr_scores):.2f} dB")
print("Avg SSIM   :", f"{np.mean(ssim_scores):.4f}")
print()
print("Saved files:")
print("/kaggle/working/mae_final_model.pt")
print("/kaggle/working/training_curves.png")
print("/kaggle/working/reconstruction_samples.png")